# HRM + Full BPTT (Left Column, Bottom Cell)
### Paper 2: Does Hierarchy Matter? Fixing the Gradient to Test Architecture

hidden=384, H_layers=4, L_layers=4, H_cycles=2, L_cycles=2, 500+500 examples, 10000 epochs
~8h on 2xT4 (full BPTT is slower)

## 1. Setup

In [1]:
!pip install -q einops coolname argdantic huggingface_hub hydra-core omegaconf pydantic wandb tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.9 MB/s eta 0:00:00


In [2]:
import os, sys, yaml, json
import torch
import numpy as np
from pathlib import Path
print(f'PyTorch: {torch.__version__}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
GPU_COUNT = torch.cuda.device_count()

PyTorch: 2.10.0+cu128
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [3]:
REPO_DIR = '/kaggle/working/HRM'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sapientinc/HRM.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

Cloning into '/kaggle/working/HRM'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 68 (delta 13), reused 10 (delta 10), pack-reused 36 (from 2)
Receiving objects: 100% (68/68), 141.52 KiB | 2.72 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Working dir: /kaggle/working/HRM


## 2. Build Dataset

In [4]:
!python dataset/build_sudoku_dataset.py --subsample-size 500 --num-aug 500 --output-dir data/sudoku-extreme-1k

train.csv: 100%|██████████████████████████████| 719M/719M [00:03<00:00, 186MB/s]
100%|█████████████████████████████████████████| 500/500 [00:28<00:00, 17.51it/s]
test.csv: 100%|████████████████████████████| 79.4M/79.4M [00:01<00:00, 56.2MB/s]
100%|██████████████████████████████| 422786/422786 [00:00<00:00, 1685155.98it/s]


## 3. Patches (AdamW + periodic print + eval limit)

In [5]:
# Patch pretrain.py: AdamW swap + periodic print + limit eval to 2K puzzles
!git -C {REPO_DIR} checkout -- pretrain.py
with open(f"{REPO_DIR}/pretrain.py", "r") as f:
    src = f.read()

# 1. AdamW swap
src = src.replace(
    "from adam_atan2 import AdamATan2",
    "from torch.optim import AdamW as AdamATan2  # T4 compat"
)

# 2. Periodic print every 200 steps
old_log = "                wandb.log(metrics, step=train_state.step)"
new_log = (
    "                if train_state.step % 200 == 0:\n"
    "                    print(f\"[Step {train_state.step}/{train_state.total_steps}] "
    "lm_loss={metrics.get('train/lm_loss',0):.2f} "
    "acc={metrics.get('train/accuracy',0):.3f} "
    "exact={metrics.get('train/exact_accuracy',0):.4f} "
    "steps={metrics.get('train/steps',0):.1f}\", flush=True)\n"
    "                wandb.log(metrics, step=train_state.step)"
)
src = src.replace(old_log, new_log)

# 3. Limit eval to 2048 test puzzles (8 batches x 256) — saves 1-2h
src = src.replace(
    "        carry = None\n",
    "        carry = None\n        processed_batches = 0\n"
)
src = src.replace(
    "            metric_global_batch_size[set_id] += global_batch_size",
    "            metric_global_batch_size[set_id] += global_batch_size\n"
    "            processed_batches += 1\n"
    "            if processed_batches >= 8:\n"
    "                break"
)

# 4. Add import json (needed for resume)
src = src.replace(
    "import yaml\nimport shutil",
    "import yaml\nimport json\nimport shutil"
)

# 5. Save step info alongside checkpoint (for resume)
old_save = (
    "    os.makedirs(config.checkpoint_path, exist_ok=True)\n"
    "    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f\"step_{train_state.step}\"))"
)
new_save = (
    "    os.makedirs(config.checkpoint_path, exist_ok=True)\n"
    "    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f\"step_{train_state.step}\"))\n"
    "    # Save step info for crash recovery\n"
    "    with open(os.path.join(config.checkpoint_path, \"resume_info.json\"), \"w\") as _f:\n"
    "        json.dump({\"step\": train_state.step}, _f)"
)
src = src.replace(old_save, new_save)

# 6. Resume from checkpoint if available (in init_train_state)
old_init = (
    "    # Model\n"
    "    model, optimizers, optimizer_lrs = create_model(config, train_metadata, world_size=world_size)\n"
    "\n"
    "    return TrainState(\n"
    "        step=0,"
)
new_init = (
    "    # Model\n"
    "    model, optimizers, optimizer_lrs = create_model(config, train_metadata, world_size=world_size)\n"
    "\n"
    "    # Resume from checkpoint if available\n"
    "    resume_step = 0\n"
    "    if config.checkpoint_path is not None and os.path.exists(config.checkpoint_path):\n"
    "        resume_file = os.path.join(config.checkpoint_path, \"resume_info.json\")\n"
    "        if os.path.exists(resume_file):\n"
    "            with open(resume_file) as _f:\n"
    "                resume_step = json.load(_f)[\"step\"]\n"
    "            ckpt_path = os.path.join(config.checkpoint_path, f\"step_{resume_step}\")\n"
    "            if os.path.exists(ckpt_path):\n"
    "                model.load_state_dict(torch.load(ckpt_path, map_location=\"cuda\"))\n"
    "                print(f\"Resumed from step {resume_step}\")\n"
    "\n"
    "    return TrainState(\n"
    "        step=resume_step,"
)
src = src.replace(old_init, new_init)

# 7. Periodic checkpoint every 500 steps (crash recovery)
old_progress = (
    "                progress_bar.update(train_state.step - progress_bar.n)  # type: ignore"
)
new_progress = (
    "                progress_bar.update(train_state.step - progress_bar.n)  # type: ignore\n"
    "                if train_state.step % 500 == 0:\n"
    "                    save_train_state(config, train_state)"
)
src = src.replace(old_progress, new_progress)

with open(f"{REPO_DIR}/pretrain.py", "w") as f:
    f.write(src)
print("Patched: AdamW + periodic print + eval limit + resume + checkpoint every 500 steps.")

# 8. Patch layers.py: fallback to SDPA when flash-attn not installed
LAYERS_FILE = f"{REPO_DIR}/models/layers.py"
!git -C {REPO_DIR} checkout -- models/layers.py
with open(LAYERS_FILE, "r") as f:
    lyr = f.read()
old_import = (
    "try:\n"
    "    from flash_attn_interface import flash_attn_func  # type: ignore[import]\n"
    "except ImportError:\n"
    "    # Fallback to FlashAttention 2\n"
    "    from flash_attn import flash_attn_func  # type: ignore[import]"
)
new_import = (
    "try:\n"
    "    from flash_attn_interface import flash_attn_func  # type: ignore[import]\n"
    "except ImportError:\n"
    "    # Fallback to FlashAttention 2\n"
    "    try:\n"
    "        from flash_attn import flash_attn_func  # type: ignore[import]\n"
    "    except ImportError:\n"
    "        flash_attn_func = None"
)
lyr = lyr.replace(old_import, new_import)
old_attn = "        attn_output = flash_attn_func(q=query, k=key, v=value, causal=self.causal)"
new_attn = (
    "        if flash_attn_func is not None:\n"
    "            attn_output = flash_attn_func(q=query, k=key, v=value, causal=self.causal)\n"
    "        else:\n"
    "            attn_output = F.scaled_dot_product_attention(\n"
    "                query.transpose(1, 2), key.transpose(1, 2), value.transpose(1, 2),\n"
    "                is_causal=self.causal,\n"
    "            ).transpose(1, 2)"
)
lyr = lyr.replace(old_attn, new_attn)
# 9. Fix .view() -> .reshape() for non-contiguous SDPA output on T4
lyr = lyr.replace("attn_output = attn_output.view(", "attn_output = attn_output.reshape(")
with open(LAYERS_FILE, "w") as f:
    f.write(lyr)
print("Patched layers.py: SDPA fallback + contiguous fix.")

Patched: AdamW + periodic print + eval limit + resume + checkpoint every 500 steps.
Patched layers.py: SDPA fallback + contiguous fix.


## 4. Gradient Method: Full BPTT (O(T) memory)

Patch `hrm_act_v1.py` to remove `torch.no_grad()` and let gradients flow through ALL recursive steps.

In [6]:
# Patch hrm_act_v1.py: Full BPTT — gradients through all H_cycles * L_cycles steps
HRM_FILE = f"{REPO_DIR}/models/hrm/hrm_act_v1.py"
!git -C {REPO_DIR} checkout -- models/hrm/hrm_act_v1.py

with open(HRM_FILE, "r") as f:
    content = f.read()

old_forward = (
    "        # Forward iterations\n"
    "        with torch.no_grad():\n"
    "            z_H, z_L = carry.z_H, carry.z_L\n"
    "\n"
    "            for _H_step in range(self.config.H_cycles):\n"
    "                for _L_step in range(self.config.L_cycles):\n"
    "                    if not ((_H_step == self.config.H_cycles - 1) and (_L_step == self.config.L_cycles - 1)):\n"
    "                        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)\n"
    "\n"
    "                if not (_H_step == self.config.H_cycles - 1):\n"
    "                    z_H = self.H_level(z_H, z_L, **seq_info)\n"
    "\n"
    "        assert not z_H.requires_grad and not z_L.requires_grad\n"
    "\n"
    "        # 1-step grad\n"
    "        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)\n"
    "        z_H = self.H_level(z_H, z_L, **seq_info)"
)

new_forward = (
    "        # Forward iterations (full BPTT)\n"
    "        z_H, z_L = carry.z_H, carry.z_L\n"
    "\n"
    "        for _H_step in range(self.config.H_cycles):\n"
    "            for _L_step in range(self.config.L_cycles):\n"
    "                z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)\n"
    "            z_H = self.H_level(z_H, z_L, **seq_info)"
)

assert old_forward in content, "Full BPTT patch: old forward pass not found!"
content = content.replace(old_forward, new_forward)

with open(HRM_FILE, "w") as f:
    f.write(content)
print("Full BPTT patch applied to hrm_act_v1.py.")

Full BPTT patch applied to hrm_act_v1.py.


## 5. Training Config

hidden=384, H_layers=4, L_layers=4, H_cycles=2, L_cycles=2, 10000 epochs, eval every 5000

In [7]:
# Write Hydra config: cfg_pretrain.yaml
with open(f"{REPO_DIR}/config/cfg_pretrain.yaml", "w") as f:
    f.write("""defaults:
  - arch: hrm_v1
  - _self_

hydra:
  output_subdir: null

data_path: data/sudoku-extreme-1k

global_batch_size: 512
epochs: 10000
eval_interval: 10000
checkpoint_every_eval: true

lr: 1e-4
lr_min_ratio: 1.0
lr_warmup_steps: 2000

beta1: 0.9
beta2: 0.95
weight_decay: 0.1
puzzle_emb_weight_decay: 0.1
puzzle_emb_lr: 1e-2

seed: 0
project_name: HRM-FullBP-Final
run_name: hrm_fullbp_final
checkpoint_path: checkpoints/hrm_fullbp_final
""")

# Write Hydra config: arch/hrm_v1.yaml
with open(f"{REPO_DIR}/config/arch/hrm_v1.yaml", "w") as f:
    f.write("""name: hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1
loss:
  name: losses@ACTLossHead
  loss_type: stablemax_cross_entropy

halt_exploration_prob: 0.1
halt_max_steps: 16

H_cycles: 2
L_cycles: 2

H_layers: 4
L_layers: 4

hidden_size: 384
num_heads: 8
expansion: 4

puzzle_emb_ndim: 384
pos_encodings: rope
forward_dtype: bfloat16
""")

print("Config files written.")

Config files written.


## 6. Launch Training

In [8]:
%env WANDB_MODE=disabled
%env DISABLE_COMPILE=1
%env TQDM_POSITION=-1
print(f"Launching training with {GPU_COUNT} GPUs (DDP)...\n")
!torchrun --nproc_per_node={GPU_COUNT} pretrain.py

env: WANDB_MODE=disabled
env: DISABLE_COMPILE=1
env: TQDM_POSITION=-1
Launching training with 2 GPUs (DDP)...

W0531 07:31:09.393000 89 torch/distributed/run.py:852] 
W0531 07:31:09.393000 89 torch/distributed/run.py:852] *****************************************
W0531 07:31:09.393000 89 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0531 07:31:09.393000 89 torch/distributed/run.py:852] *****************************************
[Rank 1, World Size 2]: Epoch 0

  0%|                                                  | 0/9765 [00:00<?, ?it/s][Rank 0, World Size 2]: Epoch 0

  2%|▊                                    | 199/9765 [13:23<10:45:34,  4.05s/it][Step 200/9765] lm_loss=2.18 acc=0.000 exact=0.0000 steps=0.0

  4%|█▌                                   | 399/9765 [26:53<10:32:59,  4.06s/it][Step

## 7. Results

| | Dual L/H (HRM) | Single flat (TRM) |
|---|---|---|
| 1-step (O1) | TBD | 2.2% (done) |
| Full BP (OT) | **THIS RUN** | 18.9% (done) |

Checkpoints at epochs 5000, 10000. Run eval cell below.

In [9]:
# Load checkpoint and evaluate
import glob as _glob, json as _json, importlib
from tqdm import tqdm
from models.losses import ACTLossHead
from utils.functions import load_model_class
importlib.reload(__import__("models.hrm.hrm_act_v1", fromlist=["*"]))

checkpoint_dir = "checkpoints/hrm_fullbp_final"
ckpts = sorted(_glob.glob(f"{checkpoint_dir}/step_*"))
if not ckpts:
    print("No checkpoints found.")
else:
    print(f"Checkpoints: {len(ckpts)}")
    with open("data/sudoku-extreme-1k/test/dataset.json") as f:
        meta = _json.load(f)
    model_cfg = dict(
        batch_size=256, vocab_size=meta["vocab_size"], seq_len=meta["seq_len"],
        num_puzzle_identifiers=meta["num_puzzle_identifiers"], causal=False,
        halt_exploration_prob=0.1, halt_max_steps=16,
        H_cycles=2, L_cycles=2, H_layers=4, L_layers=4,
        hidden_size=384, num_heads=8, expansion=4,
        puzzle_emb_ndim=384, pos_encodings="rope", forward_dtype="bfloat16"
    )
    model_cls = load_model_class("hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1")
    with torch.device("cuda"):
        model = ACTLossHead(model_cls(model_cfg), "stablemax_cross_entropy")
    state = torch.load(ckpts[-1], map_location="cuda")
    model.load_state_dict(state, strict=False)
    model.cuda()
    print(f"Loaded: {os.path.basename(ckpts[-1])}")
    SUBSET = 2000
    total_correct, total_tokens, total_exact, total_examples = 0, 0, 0, 0
    model.eval()
    with torch.inference_mode():
        for split in meta["sets"]:
            labels = np.load(f"data/sudoku-extreme-1k/test/{split}__labels.npy", mmap_mode="r")
            inputs_np = np.load(f"data/sudoku-extreme-1k/test/{split}__inputs.npy", mmap_mode="r")
            pids_np = np.load(f"data/sudoku-extreme-1k/test/{split}__puzzle_identifiers.npy", mmap_mode="r")
            n_eval = min(len(labels), SUBSET)
            pbar = tqdm(total=n_eval, desc="Eval", unit="puzzles", position=0, leave=True)
            for i in range(0, n_eval, 128):
                end = min(i + 128, n_eval)
                with torch.device("cuda"):
                    batch = {
                        "inputs": torch.tensor(inputs_np[i:end]),
                        "labels": torch.tensor(labels[i:end]),
                        "puzzle_identifiers": torch.tensor(pids_np[i:end])
                    }
                    carry = model.initial_carry(batch)
                while True:
                    carry, _, _, outputs, all_finish = model(carry=carry, batch=batch, return_keys=["logits"])
                    if all_finish:
                        break
                logits = outputs["logits"]
                pred_tokens = torch.argmax(logits, dim=-1)
                mask = batch["labels"] != 0
                total_correct += ((pred_tokens == batch["labels"]) & mask).sum().item()
                total_tokens += mask.sum().item()
                total_exact += ((pred_tokens == batch["labels"]) | ~mask).all(dim=-1).sum().item()
                total_examples += (end - i)
                pbar.update(end - i)
            pbar.close()
    print(f"\n=== RESULTS ===\nToken accuracy:  {total_correct / total_tokens * 100:.1f}%\nExact accuracy:  {total_exact / total_examples * 100:.1f}%\nTest examples:   {total_examples}")

Checkpoints: 20
Loaded: step_9765


Eval: 100%|██████████| 2000/2000 [02:05<00:00, 16.00puzzles/s]


=== RESULTS ===
Token accuracy:  69.3%
Exact accuracy:  8.7%
Test examples:   2000
